# Results Monitor Notebook

Use this notebook to review pipeline output, pre-LLM survey behavior, and accuracy against a labeled file.

Recommended flow:
1. Set `RESULTS_PATH` and optional `LABEL_PATH`.
2. Run all cells top to bottom.
3. Inspect the summary, bucket breakdown, and error tables.
4. Use the final drilldown cell for any single name.

In [39]:
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

try:
    import matplotlib.pyplot as plt
    PLOTTING = True
except Exception:
    PLOTTING = False

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)


In [40]:
# Configure these paths first.
RESULTS_PATH = Path("data/results_partial.csv")
LABEL_PATH: Optional[Path] = None
LABEL_SHEET: Optional[str] = None
NAME_COLUMN = "name"

# If your labeled file uses a specific truth column, set it here.
# Leave as None to auto-detect.
TRUTH_COLUMN: Optional[str] = None

# Optional side-by-side comparison.
# Set both paths to compare a baseline run vs a candidate run on a digestible sample.
BASELINE_RESULTS_PATH: Optional[Path] = None
CANDIDATE_RESULTS_PATH: Optional[Path] = None
COMPARE_SAMPLE_SIZE = 100
COMPARE_SAMPLE_SEED = 42

# Explorer settings.
DETAIL_NAME = "John B. Fenn"
TOP_N_ERRORS = 25

print(f"RESULTS_PATH = {RESULTS_PATH}")
print(f"LABEL_PATH   = {LABEL_PATH}")
print(f"BASELINE     = {BASELINE_RESULTS_PATH}")
print(f"CANDIDATE    = {CANDIDATE_RESULTS_PATH}")


RESULTS_PATH = data\results_partial.csv
LABEL_PATH   = None
BASELINE     = None
CANDIDATE    = None


In [ ]:
def load_table(path: Path, sheet: Optional[str] = None) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".xlsx", ".xls"}:
        data = pd.read_excel(path, sheet_name=sheet)
        if isinstance(data, dict):
            for df in data.values():
                if isinstance(df, pd.DataFrame) and not df.empty:
                    return df
            return next(iter(data.values()))
        return data
    raise ValueError(f"Unsupported file type: {suffix}")


def normalize_connected(series: pd.Series) -> pd.Series:
    values = series.fillna("").astype(str).str.strip().str.lower()
    mapping = {
        "y": True,
        "yes": True,
        "true": True,
        "1": True,
        "connected": True,
        "n": False,
        "no": False,
        "false": False,
        "0": False,
        "not_connected": False,
    }
    return values.map(mapping)


def detect_truth_column(df: pd.DataFrame) -> Optional[str]:
    candidates = [
        TRUTH_COLUMN,
        "actual_connected",
        "ground_truth",
        "truth_connected",
        "is_connected",
        "is_purdue",
        "expected_connected",
        "manual_connected",
        "connected_true",
        "label",
        "truth",
    ]
    for candidate in candidates:
        if candidate and candidate in df.columns:
            return candidate
    return None


def prepare_results(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if NAME_COLUMN not in out.columns:
        raise ValueError(f"Missing name column: {NAME_COLUMN}")

    out[NAME_COLUMN] = out[NAME_COLUMN].astype(str).str.strip()
    if "connected" in out.columns:
        out["pred_connected"] = normalize_connected(out["connected"])
    elif "verdict" in out.columns:
        out["pred_connected"] = out["verdict"].astype(str).str.strip().str.lower().eq("connected")
    else:
        out["pred_connected"] = pd.Series([pd.NA] * len(out), dtype="boolean")

    for col in [
        "pre_llm_bucket",
        "pre_llm_stage",
        "pre_llm_score",
        "pre_llm_summary",
        "pre_llm_reason_codes",
        "pre_llm_used_rescue_query",
        "verification_status",
        "confidence",
        "verdict",
        "relationship_type",
        "summary",
        "verification_detail",
        "primary_source",
    ]:
        if col not in out.columns:
            out[col] = pd.NA

    if "pre_llm_score" in out.columns:
        out["pre_llm_score_num"] = pd.to_numeric(out["pre_llm_score"], errors="coerce")
    else:
        out["pre_llm_score_num"] = np.nan

    out["row_id"] = np.arange(1, len(out) + 1)
    return out


def merge_labels(results_df: pd.DataFrame, labels_df: Optional[pd.DataFrame]) -> pd.DataFrame:
    if labels_df is None:
        out = results_df.copy()
        truth_col = detect_truth_column(out)
        if truth_col:
            out["truth_connected"] = normalize_connected(out[truth_col])
        else:
            out["truth_connected"] = pd.Series([pd.NA] * len(out), dtype="boolean")
        return out

    labels = labels_df.copy()
    if NAME_COLUMN not in labels.columns:
        raise ValueError(f"Label file missing name column: {NAME_COLUMN}")
    labels[NAME_COLUMN] = labels[NAME_COLUMN].astype(str).str.strip()
    truth_col = detect_truth_column(labels)
    if not truth_col:
        raise ValueError("Could not auto-detect truth column in label file. Set TRUTH_COLUMN manually.")

    labels = labels[[NAME_COLUMN, truth_col]].drop_duplicates(subset=[NAME_COLUMN], keep="first").copy()
    labels["truth_connected"] = normalize_connected(labels[truth_col])
    merged = results_df.merge(labels[[NAME_COLUMN, "truth_connected"]], on=NAME_COLUMN, how="left")
    return merged


def accuracy_summary(df: pd.DataFrame) -> pd.DataFrame:
    eligible = df[df["truth_connected"].notna() & df["pred_connected"].notna()].copy()
    if eligible.empty:
        return pd.DataFrame()

    tp = int(((eligible["pred_connected"] == True) & (eligible["truth_connected"] == True)).sum())
    tn = int(((eligible["pred_connected"] == False) & (eligible["truth_connected"] == False)).sum())
    fp = int(((eligible["pred_connected"] == True) & (eligible["truth_connected"] == False)).sum())
    fn = int(((eligible["pred_connected"] == False) & (eligible["truth_connected"] == True)).sum())
    total = len(eligible)
    accuracy = (tp + tn) / total if total else np.nan
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return pd.DataFrame(
        {
            "metric": ["rows_with_truth", "tp", "tn", "fp", "fn", "accuracy", "precision", "recall", "specificity"],
            "value": [total, tp, tn, fp, fn, accuracy, precision, recall, specificity],
        }
    )


def bucket_summary(df: pd.DataFrame) -> pd.DataFrame:
    bucket_cols = ["pre_llm_stage", "pre_llm_bucket", "pred_connected", "verification_status", "confidence"]
    existing = [c for c in bucket_cols if c in df.columns]
    out = (
        df.groupby(existing, dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    return out


def show_metric_cards(df: pd.DataFrame):
    total = len(df)
    connected = int(df["pred_connected"].eq(True).sum()) if "pred_connected" in df.columns else 0
    not_connected = int(df["pred_connected"].eq(False).sum()) if "pred_connected" in df.columns else 0
    hard_no = int(df.get("pre_llm_bucket", pd.Series(dtype=str)).fillna("").eq("hard_no").sum())
    borderline = int(df.get("pre_llm_bucket", pd.Series(dtype=str)).fillna("").eq("borderline").sum())
    plausible = int(df.get("pre_llm_bucket", pd.Series(dtype=str)).fillna("").eq("plausible").sum())
    rejected_fast = int(df.get("pre_llm_stage", pd.Series(dtype=str)).fillna("").eq("reject_fast").sum())
    rejected_after_rescue = int(df.get("pre_llm_stage", pd.Series(dtype=str)).fillna("").eq("reject_after_rescue").sum())
    rescue_used = int(df.get("pre_llm_used_rescue_query", pd.Series(dtype=str)).fillna("").astype(str).str.upper().eq("Y").sum())

    cards = pd.DataFrame(
        {
            "metric": ["total_rows", "connected", "not_connected", "hard_no", "borderline", "plausible", "rejected_fast", "rejected_after_rescue", "rescue_used"],
            "value": [total, connected, not_connected, hard_no, borderline, plausible, rejected_fast, rejected_after_rescue, rescue_used],
        }
    )
    display(cards)


def review_columns(df: pd.DataFrame):
    preferred = [
        NAME_COLUMN,
        "verdict",
        "connected",
        "relationship_type",
        "confidence",
        "verification_status",
        "pre_llm_bucket",
        "pre_llm_stage",
        "pre_llm_score",
        "pre_llm_used_rescue_query",
        "pre_llm_summary",
        "summary",
        "primary_source",
        "truth_connected",
    ]
    return [c for c in preferred if c in df.columns]


def build_comparison_frame(
    baseline_results: pd.DataFrame,
    candidate_results: pd.DataFrame,
    labels_df: Optional[pd.DataFrame] = None,
    sample_size: int = 100,
    sample_seed: int = 42,
) -> pd.DataFrame:
    baseline = prepare_results(baseline_results).copy()
    candidate = prepare_results(candidate_results).copy()

    keep_cols = [
        NAME_COLUMN,
        "verdict",
        "connected",
        "relationship_type",
        "confidence",
        "verification_status",
        "pre_llm_bucket",
        "pre_llm_score",
        "pre_llm_summary",
        "pre_llm_reason_codes",
        "pre_llm_used_rescue_query",
        "summary",
        "primary_source",
        "pred_connected",
    ]
    baseline = baseline[[c for c in keep_cols if c in baseline.columns]].drop_duplicates(subset=[NAME_COLUMN], keep="first")
    candidate = candidate[[c for c in keep_cols if c in candidate.columns]].drop_duplicates(subset=[NAME_COLUMN], keep="first")

    merged = baseline.merge(candidate, on=NAME_COLUMN, how="inner", suffixes=("_base", "_cand"))
    if merged.empty:
        return merged

    if labels_df is not None:
        labels = labels_df.copy()
        labels[NAME_COLUMN] = labels[NAME_COLUMN].astype(str).str.strip()
        truth_col = detect_truth_column(labels)
        if truth_col:
            labels = labels[[NAME_COLUMN, truth_col]].drop_duplicates(subset=[NAME_COLUMN], keep="first")
            labels["truth_connected"] = normalize_connected(labels[truth_col])
            merged = merged.merge(labels[[NAME_COLUMN, "truth_connected"]], on=NAME_COLUMN, how="left")
        else:
            merged["truth_connected"] = pd.Series([pd.NA] * len(merged), dtype="boolean")
    else:
        merged["truth_connected"] = pd.Series([pd.NA] * len(merged), dtype="boolean")

    merged["verdict_changed"] = merged["verdict_base"].astype(str) != merged["verdict_cand"].astype(str)
    merged["bucket_changed"] = merged["pre_llm_bucket_base"].astype(str) != merged["pre_llm_bucket_cand"].astype(str)
    merged["score_delta"] = pd.to_numeric(merged["pre_llm_score_cand"], errors="coerce") - pd.to_numeric(merged["pre_llm_score_base"], errors="coerce")

    if merged["truth_connected"].notna().any():
        base_ok = merged["pred_connected_base"] == merged["truth_connected"]
        cand_ok = merged["pred_connected_cand"] == merged["truth_connected"]
        merged["accuracy_movement"] = np.select(
            [
                (~base_ok) & (cand_ok),
                (base_ok) & (~cand_ok),
                (base_ok) & (cand_ok),
                (~base_ok) & (~cand_ok),
            ],
            ["improved", "regressed", "stayed_correct", "stayed_wrong"],
            default="unknown",
        )
    else:
        merged["accuracy_movement"] = "unknown"

    changed = merged[merged["verdict_changed"] | merged["bucket_changed"]].copy()
    population = changed if not changed.empty else merged
    if len(population) > sample_size:
        sample = population.sample(n=sample_size, random_state=sample_seed)
    else:
        sample = population.copy()
    return sample.sort_values(["accuracy_movement", "verdict_changed", "bucket_changed", NAME_COLUMN], ascending=[True, False, False, True])


In [42]:
results_raw = load_table(RESULTS_PATH)
labels_raw = load_table(LABEL_PATH, sheet=LABEL_SHEET) if LABEL_PATH else None

results_df = prepare_results(results_raw)
monitor_df = merge_labels(results_df, labels_raw)

display(Markdown(f"## Loaded {len(monitor_df):,} rows"))
display(monitor_df[review_columns(monitor_df)].head(10))


## Loaded 1 rows

,name,verdict,connected,relationship_type,confidence,verification_status,pre_llm_bucket,pre_llm_score,pre_llm_used_rescue_query,pre_llm_summary,summary,primary_source,truth_connected
0,John B. Fenn,connected,Y,Attended,high,verified,<NA>,<NA>,<NA>,<NA>,"John B. Fenn attended Purdue University for summer coursework in physical chemistry, establishing a past student relationship.",https://link.springer.com/content/pdf/10.1007/s13361-011-0136-6.pdf,<NA>


In [43]:
display(Markdown("## Run Summary"))
show_metric_cards(monitor_df)

if "pre_llm_bucket" in monitor_df.columns and monitor_df["pre_llm_bucket"].notna().any():
    display(Markdown("### Pre-LLM Bucket Breakdown"))
    bucket_df = bucket_summary(monitor_df)
    display(bucket_df.head(20))

if PLOTTING and "pre_llm_bucket" in monitor_df.columns and monitor_df["pre_llm_bucket"].notna().any():
    counts = monitor_df["pre_llm_bucket"].fillna("(missing)").value_counts(dropna=False)
    ax = counts.plot(kind="bar", figsize=(7, 4), title="Pre-LLM Survey Buckets")
    ax.set_xlabel("bucket")
    ax.set_ylabel("count")
    plt.show()


## Run Summary

,metric,value
0,total_rows,1
1,connected,1
2,not_connected,0
3,hard_no,0
4,borderline,0
5,plausible,0
6,rescue_used,0


In [44]:
display(Markdown("## Accuracy Summary"))
acc_df = accuracy_summary(monitor_df)
if acc_df.empty:
    print("No truth labels found. Set LABEL_PATH or TRUTH_COLUMN to calculate accuracy.")
else:
    display(acc_df)
    reviewed = monitor_df[monitor_df["truth_connected"].notna() & monitor_df["pred_connected"].notna()].copy()
    reviewed["confusion"] = np.select(
        [
            (reviewed["pred_connected"] == True) & (reviewed["truth_connected"] == True),
            (reviewed["pred_connected"] == False) & (reviewed["truth_connected"] == False),
            (reviewed["pred_connected"] == True) & (reviewed["truth_connected"] == False),
            (reviewed["pred_connected"] == False) & (reviewed["truth_connected"] == True),
        ],
        ["TP", "TN", "FP", "FN"],
        default="?",
    )
    display(reviewed["confusion"].value_counts().rename_axis("bucket").reset_index(name="count"))


## Accuracy Summary

No truth labels found. Set LABEL_PATH or TRUTH_COLUMN to calculate accuracy.


In [45]:
display(Markdown("## Review Tables"))

base_cols = review_columns(monitor_df)
has_truth = monitor_df["truth_connected"].notna().any() and monitor_df["pred_connected"].notna().any()
reviewed = monitor_df[monitor_df["truth_connected"].notna() & monitor_df["pred_connected"].notna()].copy() if has_truth else pd.DataFrame()

if has_truth:
    false_negatives = reviewed[(reviewed["truth_connected"] == True) & (reviewed["pred_connected"] == False)].copy()
    false_positives = reviewed[(reviewed["truth_connected"] == False) & (reviewed["pred_connected"] == True)].copy()

    display(Markdown(f"### False Negatives ({len(false_negatives)})"))
    display(false_negatives.sort_values(["pre_llm_bucket", "pre_llm_score_num"], ascending=[True, True])[base_cols].head(TOP_N_ERRORS))

    display(Markdown(f"### False Positives ({len(false_positives)})"))
    display(false_positives.sort_values(["pre_llm_bucket", "pre_llm_score_num"], ascending=[True, False])[base_cols].head(TOP_N_ERRORS))

    hard_skipped_positives = reviewed[(reviewed["truth_connected"] == True) & (reviewed.get("pre_llm_bucket", pd.Series(index=reviewed.index)).fillna("") == "hard_no")].copy()
    display(Markdown(f"### True Positives Lost at Pre-LLM Survey ({len(hard_skipped_positives)})"))
    display(hard_skipped_positives[base_cols].head(TOP_N_ERRORS))
else:
    print("No truth labels available, so false-positive / false-negative tables are skipped.")

borderline_cases = monitor_df[monitor_df.get("pre_llm_bucket", pd.Series(index=monitor_df.index)).fillna("") == "borderline"].copy()
display(Markdown(f"### Borderline Cases ({len(borderline_cases)})"))
display(borderline_cases.sort_values("pre_llm_score_num", ascending=True)[base_cols].head(TOP_N_ERRORS))

hard_no_cases = monitor_df[monitor_df.get("pre_llm_bucket", pd.Series(index=monitor_df.index)).fillna("") == "hard_no"].copy()
display(Markdown(f"### Hard No Cases ({len(hard_no_cases)})"))
display(hard_no_cases.sort_values("pre_llm_score_num", ascending=True)[base_cols].head(TOP_N_ERRORS))


## Review Tables

No truth labels available, so false-positive / false-negative tables are skipped.


### Borderline Cases (0)

,name,verdict,connected,relationship_type,confidence,verification_status,pre_llm_bucket,pre_llm_score,pre_llm_used_rescue_query,pre_llm_summary,summary,primary_source,truth_connected


### Hard No Cases (0)

,name,verdict,connected,relationship_type,confidence,verification_status,pre_llm_bucket,pre_llm_score,pre_llm_used_rescue_query,pre_llm_summary,summary,primary_source,truth_connected


In [46]:
display(Markdown("## Baseline vs Candidate Comparison"))
if not BASELINE_RESULTS_PATH or not CANDIDATE_RESULTS_PATH:
    print("Set BASELINE_RESULTS_PATH and CANDIDATE_RESULTS_PATH to compare two runs on a 100-name sample.")
else:
    baseline_raw = load_table(BASELINE_RESULTS_PATH)
    candidate_raw = load_table(CANDIDATE_RESULTS_PATH)
    compare_df = build_comparison_frame(
        baseline_raw,
        candidate_raw,
        labels_df=labels_raw,
        sample_size=COMPARE_SAMPLE_SIZE,
        sample_seed=COMPARE_SAMPLE_SEED,
    )
    if compare_df.empty:
        print("No overlapping names found between baseline and candidate result files.")
    else:
        print(f"Showing {len(compare_df):,} names (sample size target: {COMPARE_SAMPLE_SIZE}).")
        summary_cols = ["accuracy_movement", "verdict_changed", "bucket_changed"]
        display(compare_df[summary_cols].value_counts(dropna=False).rename_axis(summary_cols).reset_index(name="count"))


## Baseline vs Candidate Comparison

Set BASELINE_RESULTS_PATH and CANDIDATE_RESULTS_PATH to compare two runs on a 100-name sample.


In [47]:
if 'compare_df' not in globals() or compare_df.empty:
    print("Run the comparison cell above first.")
else:
    compare_cols = [
        NAME_COLUMN,
        "truth_connected",
        "accuracy_movement",
        "verdict_base",
        "verdict_cand",
        "relationship_type_base",
        "relationship_type_cand",
        "pre_llm_bucket_base",
        "pre_llm_bucket_cand",
        "pre_llm_score_base",
        "pre_llm_score_cand",
        "score_delta",
        "pre_llm_used_rescue_query_base",
        "pre_llm_used_rescue_query_cand",
        "summary_base",
        "summary_cand",
    ]
    compare_cols = [c for c in compare_cols if c in compare_df.columns]
    display(Markdown("### 100-Name Comparison Sample"))
    display(compare_df[compare_cols])

    improved = compare_df[compare_df["accuracy_movement"] == "improved"].copy()
    regressed = compare_df[compare_df["accuracy_movement"] == "regressed"].copy()
    changed = compare_df[compare_df["verdict_changed"] | compare_df["bucket_changed"]].copy()

    display(Markdown(f"### Improved Cases ({len(improved)})"))
    display(improved[compare_cols].head(TOP_N_ERRORS))

    display(Markdown(f"### Regressed Cases ({len(regressed)})"))
    display(regressed[compare_cols].head(TOP_N_ERRORS))

    display(Markdown(f"### Changed Cases ({len(changed)})"))
    display(changed[compare_cols].head(TOP_N_ERRORS))


Run the comparison cell above first.


In [48]:
display(Markdown(f"## Name Drilldown: {DETAIL_NAME}"))
detail = monitor_df[monitor_df[NAME_COLUMN].astype(str).str.lower() == str(DETAIL_NAME).strip().lower()].copy()
if detail.empty:
    print("Name not found in loaded results.")
else:
    row = detail.iloc[0]
    detail_cols = [
        NAME_COLUMN,
        "verdict",
        "connected",
        "relationship_type",
        "confidence",
        "verification_status",
        "pre_llm_bucket",
        "pre_llm_score",
        "pre_llm_reason_codes",
        "pre_llm_used_rescue_query",
        "pre_llm_summary",
        "summary",
        "verification_detail",
        "primary_source",
        "truth_connected",
    ]
    detail_cols = [c for c in detail_cols if c in detail.index]
    display(detail[detail_cols])
    for field in ["pre_llm_summary", "summary", "verification_detail", "primary_source"]:
        if field in row.index and pd.notna(row[field]):
            display(Markdown(f"**{field}**: {row[field]}"))


## Name Drilldown: John B. Fenn

""
0


**summary**: John B. Fenn attended Purdue University for summer coursework in physical chemistry, establishing a past student relationship.

**verification_detail**: "He earned his bachelor's degree from Berea College ... with the assistance of summer classes in organic chemistry at the University of Iowa, and physical chemistry at Purdue." (Source #9) This shows Fenn studied physical chemistry at Purdue University as a student, though he did not earn a degree there.

**primary_source**: https://link.springer.com/content/pdf/10.1007/s13361-011-0136-6.pdf